In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    mean_absolute_error, 
    mean_squared_error, 
    r2_score
)

In [2]:
df_flights_limpas_0 = pd.read_csv("data/flights_limpas.csv")

In [3]:
df_flights_limpas_0

,SCHEDULED_DEPARTURE,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_DELAY,BOL_DELAYED
0,5,-11.0,21.0,15.0,194.0,169.0,1448,404.0,4.0,430,-22.0,0
1,10,-8.0,12.0,14.0,279.0,263.0,2330,737.0,4.0,750,-9.0,0
2,20,-2.0,16.0,34.0,293.0,266.0,2296,800.0,11.0,806,5.0,1
3,20,-5.0,15.0,30.0,281.0,258.0,2342,748.0,8.0,805,-9.0,0
4,25,-1.0,11.0,35.0,215.0,199.0,1448,254.0,5.0,320,-21.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
4755635,2359,-4.0,22.0,17.0,298.0,272.0,2611,749.0,4.0,819,-26.0,0
4755636,2359,-4.0,17.0,12.0,215.0,195.0,1617,427.0,3.0,446,-16.0,0
4755637,2359,-9.0,17.0,7.0,222.0,197.0,1598,424.0,8.0,440,-8.0,0
4755638,2359,-6.0,10.0,3.0,157.0,144.0,1189,327.0,3.0,340,-10.0,0


In [4]:
df_flights_limpas = df_flights_limpas_0.drop(columns=['ARRIVAL_DELAY'])

In [5]:
df_flights_limpas_0.shape

(4755640, 12)

In [6]:
# Verificando a presença de valores NaN em cada coluna do DataFrame
nan_counts = df_flights_limpas_0.isna().sum()
print("Valores NaN por coluna:\n", nan_counts)
print("\nTotal de valores NaN no DataFrame:", df_flights_limpas_0.isna().sum().sum())

Valores NaN por coluna:
 SCHEDULED_DEPARTURE         0
DEPARTURE_DELAY         86153
TAXI_OUT                89047
WHEELS_OFF              89047
ELAPSED_TIME           105071
AIR_TIME               105071
DISTANCE                    0
WHEELS_ON               92513
TAXI_IN                 92513
SCHEDULED_ARRIVAL           0
ARRIVAL_DELAY          105071
BOL_DELAYED                 0
dtype: int64

Total de valores NaN no DataFrame: 764486


In [7]:
# Removendo todas as linhas que contém pelo menos um valor NaN
df_flights_limpas = df_flights_limpas_0.dropna()

In [8]:
df_flights_limpas.shape

(4650569, 12)

In [9]:
print("Número de linhas removidas:", df_flights_limpas_0.shape[0] - df_flights_limpas.shape[0])

Número de linhas removidas: 105071


In [10]:
# Definindo X e y
X = df_flights_limpas.drop(columns=['BOL_DELAYED'])
y = df_flights_limpas['BOL_DELAYED']

# Separando em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Treinando o modelo KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

# Fazendo previsões
y_pred = knn.predict(X_test)

# Avaliando o modelo
print("Acurácia:", accuracy_score(y_test, y_pred))
print("\nRelatório de classificação:\n", classification_report(y_test, y_pred))

Acurácia: 0.9424715680013418

Relatório de classificação:
               precision    recall  f1-score   support

           0       0.95      0.98      0.96    700180
           1       0.92      0.84      0.88    229934

    accuracy                           0.94    930114
   macro avg       0.93      0.91      0.92    930114
weighted avg       0.94      0.94      0.94    930114



In [11]:
# Realizando validação cruzada com 5 folds
cv_scores = cross_val_score(knn, X, y, cv=5, scoring='accuracy')

print("Acurácia em cada fold da validação cruzada:", cv_scores)
print("Acurácia média da validação cruzada:", cv_scores.mean())

Acurácia em cada fold da validação cruzada: [0.92440819 0.93040961 0.93146539 0.93580894 0.92708198]
Acurácia média da validação cruzada: 0.92983482177066


In [12]:
# Recarregando o DF para o modelo de regressao linear
# (mantendo as mesmas colunas usadas no fluxo atual)
df_flights_limpas_0 = pd.read_csv("data/flights_limpas.csv")
df_flights_limpas_0.drop(columns=['BOL_DELAYED'])

# Removendo todas as linhas que contem pelo menos um valor NaN
df_flights_limpas = df_flights_limpas_0.dropna()

In [13]:
# Definindo X e y para regressao linear; target = ARRIVAL_DELAY
X = df_flights_limpas.drop(columns=['ARRIVAL_DELAY']).copy()
y = df_flights_limpas['ARRIVAL_DELAY'].astype('float32')

# Features ciclicas para horario (melhoram modelos lineares em variaveis HHMM)
def hhmm_to_minutes(series):
    s = series.astype(int)
    hh = s // 100
    mm = s % 100
    return (hh * 60 + mm).astype('float32')

for col in ['SCHEDULED_DEPARTURE', 'SCHEDULED_ARRIVAL', 'WHEELS_OFF', 'WHEELS_ON']:
    minutes = hhmm_to_minutes(X[col])
    angle = 2 * np.pi * (minutes / 1440.0)
    X[f'{col}_SIN'] = np.sin(angle).astype('float32')
    X[f'{col}_COS'] = np.cos(angle).astype('float32')

# Mantem as variaveis originais em float32 para menor uso de memoria
X = X.astype('float32')

# Dividindo em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Busca automatica do melhor alpha com faixa mais ampla
alphas = np.logspace(-3, 3, 13, dtype=np.float32)
ridgecv_model = make_pipeline(
    StandardScaler(),
    RidgeCV(alphas=alphas, cv=5, scoring='neg_root_mean_squared_error')
)

# Treinamento e previsao
ridgecv_model.fit(X_train, y_train)
y_pred = ridgecv_model.predict(X_test)

# Melhor alpha encontrado
best_alpha = ridgecv_model.named_steps['ridgecv'].alpha_

# Avaliacao
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print('Modelo: RidgeCV + features ciclicas de horario')
print('Melhor alpha:', best_alpha)
print('MAE:', mae)
print('MSE:', mse)
print('RMSE:', rmse)
print('R²:', r2)

c:\Users\PC\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\linear_model\_ridge.py:215: LinAlgWarning: Ill-conditioned matrix (rcond=3.95611e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\PC\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\linear_model\_ridge.py:215: LinAlgWarning: Ill-conditioned matrix (rcond=3.95611e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\PC\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\linear_model\_ridge.py:215: LinAlgWarning: Ill-conditioned matrix (rcond=3.95611e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\PC\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\linear_model\_ridge.py:215: LinAlgWarning: Ill-conditioned matrix (rcond=3.95611e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a

Modelo: RidgeCV + features ciclicas de horario
Melhor alpha: 0.31622776
MAE: 4.388002395629883
MSE: 35.97403335571289
RMSE: 5.997835722634698
R²: 0.6900169253349304
